# Lesson 02 - Pagsusuri sa Microsoft Agent Framework

Ang **Microsoft Agent Framework (MAF)** ay isang pinag-isang balangkas para sa paggawa ng mga AI agent. Nagbibigay ito ng malinis, kompositor na arkitektura na may apat na pangunahing bahagi:

- **Client** – kumokonekta sa isang AI model endpoint at humahawak ng komunikasyon
- **Agent** – bumabalot sa isang client gamit ang mga instruksiyon at depinisyon ng kasangkapan
- **Tools** – nagpapalawak ng kakayahan ng agent gamit ang mga pasadyang function na maaaring tawagin ng modelo
- **Session** – nagpapanatili ng kasaysayan ng pag-uusap para sa multi-turn na interaksyon

Sa araling ito, gagawa tayo ng **travel booking agent** na sumusuri ng availability ng destinasyon gamit ang mga konseptong ito.


## Setup


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## Pag-unawa sa Arkitektura ng Agent Framework

Ang Microsoft Agent Framework ay sumusunod sa isang layered na arkitektura:

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **Client** – Ang `AzureAIProjectAgentProvider` ay kumokonekta sa isang Azure OpenAI deployment. Ito ang humahawak ng authentication, pag-format ng kahilingan, at pag-parse ng sagot.
2. **Agent** – Ginawa mula sa client sa pamamagitan ng `provider.create_agent()`, pinagsasama ng agent ang access sa modelo kasama ang mga tagubilin (system prompt) at mga kasangkapan.
3. **Tools** – Mga Python function na may dekorador na `@tool` na maaaring tawagin ng agent upang magsagawa ng mga aksyon o kumuha ng data.
4. **Session** – Isang `AgentSession` na object (ginawa sa pamamagitan ng `agent.create_session()`) na nag-iimbak ng kasaysayan ng pag-uusap, na nagpapahintulot ng multi-turn dialogue kung saan naaalala ng agent ang nakaraang konteksto.

Itayo natin ang bawat layer nang paisa-isa.


In [ ]:
# Create the client – this is the connection to the AI model
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Pagdaragdag ng Mga Tool gamit ang @tool Decorator

Pinahihintulutan ng mga tool ang mga ahente na gumawa ng mga aksyon lampas sa paggawa ng teksto. Kinokonvert ng `@tool` decorator ang isang karaniwang Python function sa isang bagay na maaaring tawagin ng ahente.

Pangunahing punto:
- Gamitin ang `Annotated[type, "description"]` upang maunawaan ng modelo ang bawat parameter.
- Nagiging deskripsyon ng tool na nakikita ng modelo ang docstring.
- Ang `approval_mode="never_require"` ay nangangahulugang ang tool ay tumatakbo nang awtomatiko nang walang kumpirmasyon mula sa gumagamit.


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## Paglikha ng Ahente gamit ang Mga Kasangkapan

Ngayon pinag-sama natin ang kliyente, mga tagubilin, at mga kasangkapan sa isang ahente. Ang `instructions` ay nagsisilbing system prompt — tinutukoy nito ang persona at pag-uugali ng ahente.


In [ ]:
agent = await provider.create_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## Mga Multi-Turn na Usapan gamit ang Mga Session

Ang isang `AgentSession` (na nilikha sa pamamagitan ng `agent.create_session()`) ay nagtatala ng lahat ng mga mensahe sa isang pag-uusap. Sa pamamagitan ng pagpasa ng parehong session sa bawat tawag ng `agent.run()`, may access ang agent sa buong kasaysayan ng pag-uusap at maaaring bumalik sa mga naunang mensahe.

Ipapasa namin ang `tools=[check_destination_availability]` upang ang agent ay maaaring tawagan ang aming tagasuri ng pagiging available sa bawat turn.


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## Buod

Sa araling ito sinuri mo ang apat na haligi ng Microsoft Agent Framework:

| Konsepto | Ang Iyong Napag-aralan |
|---------|------------------|
| **Kliyente** | `AzureAIProjectAgentProvider` kumokonekta sa Azure OpenAI gamit ang credential-based na auth |
| **Ahente** | `provider.create_agent()` pinagsasama ang koneksyon sa modelo kasama ang mga tagubilin at pangalan |
| **Mga Kasangkapan** | Ang `@tool` decorator ay naglalantad ng mga Python function para tawagin ng ahente |
| **Session** | `agent.create_session()` nagpapanatili ng kasaysayan ng pag-uusap sa maraming pagkakataon |

Ang mga pundasyong ito ay pinagsama-sama upang makalikha ng mga ahenteng makapaglalahad ng natural na pag-uusap, makatawag ng panlabas na mga function, at mapanatili ang konteksto — ang pundasyon para sa mas advanced na mga pattern ng ahente sa mga susunod na aralin.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Paunawa**:
Ang dokumentong ito ay isinalin gamit ang serbisyong AI na pagsasalin na [Co-op Translator](https://github.com/Azure/co-op-translator). Bagamat nagsisikap kaming maging tumpak, pakatandaan na ang awtomatikong pagsasalin ay maaaring maglaman ng mga pagkakamali o hindi pagkakatugma. Ang orihinal na dokumento sa kanyang wikang katutubo ang dapat ituring na pangunahing sanggunian. Para sa mahahalagang impormasyon, inirerekomendang gumamit ng propesyonal na pagsasaling-tao. Hindi kami mananagot sa anumang hindi pagkakaunawaan o maling interpretasyon na nagmumula sa paggamit ng pagsasaling ito.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
